In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ===============================
# 1) CARICA TRAIN DATASET
# ===============================
train_dataset = pd.read_csv("hand_dataset_train.csv")
X = train_dataset.iloc[:, 1:].values
Y = train_dataset.iloc[:, 0].values

# Splitta in train e validation (20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# ===============================
# 2) CARICA TEST DATASET
# ===============================
test_dataset = pd.read_csv("hand_dataset_test.csv")
X_test = test_dataset.iloc[:, 1:].values
y_test = test_dataset.iloc[:, 0].values

# ===============================
# 3) SCALING
# ===============================
scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# ===============================
# 4) INFO SHAPE
# ===============================
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


X_train: (21576, 42)
X_val: (5394, 42)
X_test: (2952, 42)


In [3]:
import numpy as np
classes = np.unique(y_train)
for cls in classes:
    X_cls_train = X_train[y_train == cls]
    print(f"Classe {cls}: {X_cls_train.shape[0]} campioni")


Classe A: 713 campioni
Classe B: 507 campioni
Classe C: 938 campioni
Classe D: 926 campioni
Classe E: 806 campioni
Classe F: 1023 campioni
Classe G: 986 campioni
Classe H: 966 campioni
Classe I: 966 campioni
Classe J: 370 campioni
Classe K: 1154 campioni
Classe L: 1021 campioni
Classe M: 732 campioni
Classe N: 724 campioni
Classe O: 703 campioni
Classe P: 841 campioni
Classe Q: 697 campioni
Classe R: 1022 campioni
Classe S: 903 campioni
Classe T: 774 campioni
Classe U: 694 campioni
Classe V: 990 campioni
Classe W: 923 campioni
Classe X: 970 campioni
Classe Y: 860 campioni
Classe Z: 367 campioni


In [4]:
import numpy as np
from sklearn.mixture import GaussianMixture

# Classi presenti
classes = np.unique(y_train)

# ===============================
# 2) STIMA PRIORS (con train+val)
# ===============================
priors = {}
for cls in classes:
    priors[cls] = np.sum(y_train == cls) / len(y_train)

print("Priors:", priors)

# Numero di componenti candidate
component_candidates = [1, 2, 3, 4, 5, 6, 8, 10]

# Dizionari per salvare i migliori risultati
best_components_bic = {}
best_components_aic = {}
best_components_ll  = {}
best_gmms_bic = {}
best_gmms_aic = {}
best_gmms_ll  = {}

for cls in classes:
    Xc_train = X_train[y_train == cls]
    Xc_val   = X_val[y_val == cls]

    # Inizializza variabili per confronto
    best_bic = np.inf
    best_aic = np.inf
    best_ll  = -np.inf
    best_gmm_bic = None
    best_gmm_aic = None
    best_gmm_ll  = None

    for k in component_candidates:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            random_state=42
        )
        gmm.fit(Xc_train)

        # Calcolo criteri
        bic = gmm.bic(Xc_val)
        aic = gmm.aic(Xc_val)
        ll  = gmm.score(Xc_val) * len(Xc_val)  # log-likelihood totale

        # Salva migliore BIC
        if bic < best_bic:
            best_bic = bic
            best_components_bic[cls] = k
            best_gmms_bic[cls] = gmm

        # Salva migliore AIC
        if aic < best_aic:
            best_aic = aic
            best_components_aic[cls] = k
            best_gmms_aic[cls] = gmm

        # Salva migliore log-likelihood pura
        if ll > best_ll:
            best_ll = ll
            best_components_ll[cls] = k
            best_gmms_ll[cls] = gmm

    print(f"Classe {cls}: BIC={best_components_bic[cls]}, AIC={best_components_aic[cls]}, LL={best_components_ll[cls]}")

# Ricombinazione train + val per allenamento finale
X_train_full = np.concatenate([X_train, X_val], axis=0)
y_train_full = np.concatenate([y_train, y_val], axis=0)

print("\nRicombinazione completata:")
print("X_train_full:", X_train_full.shape)


Priors: {'A': np.float64(0.033045977011494254), 'B': np.float64(0.02349833147942158), 'C': np.float64(0.04347423062662217), 'D': np.float64(0.042918057100482014), 'E': np.float64(0.03735632183908046), 'F': np.float64(0.04741379310344827), 'G': np.float64(0.0456989247311828), 'H': np.float64(0.044771968854282536), 'I': np.float64(0.044771968854282536), 'J': np.float64(0.017148683722654802), 'K': np.float64(0.05348535409714498), 'L': np.float64(0.04732109751575825), 'M': np.float64(0.0339265850945495), 'N': np.float64(0.033555802743789394), 'O': np.float64(0.03258249907304412), 'P': np.float64(0.038978494623655914), 'Q': np.float64(0.032304412309974044), 'R': np.float64(0.04736744530960326), 'S': np.float64(0.041852057842046715), 'T': np.float64(0.03587319243604004), 'U': np.float64(0.032165368928439006), 'V': np.float64(0.04588431590656285), 'W': np.float64(0.042779013718946976), 'X': np.float64(0.04495736002966259), 'Y': np.float64(0.03985910270671116), 'Z': np.float64(0.01700964034111

In [5]:
# ===============================
# 5) TRAIN FINALE DEI GMM PER CLASSE
# ===============================

final_gmms_bic = {}
final_gmms_aic = {}
final_gmms_ll  = {}

for cls in classes:

    # Recupera il numero di componenti scelto da ciascun criterio
    k_bic = best_components_bic[cls]
    k_aic = best_components_aic[cls]
    k_ll  = best_components_ll[cls]

    Xc = X_train_full[y_train_full == cls]

    # ----- BIC -----
    print(f"Addestro modello finale per classe {cls} con {k_bic} componenti (BIC)...")
    gmm_bic = GaussianMixture(
        n_components=k_bic,
        covariance_type='full',
        random_state=42
    )
    gmm_bic.fit(Xc)
    final_gmms_bic[cls] = gmm_bic

    # ----- AIC -----
    print(f"Addestro modello finale per classe {cls} con {k_aic} componenti (AIC)...")
    gmm_aic = GaussianMixture(
        n_components=k_aic,
        covariance_type='full',
        random_state=42
    )
    gmm_aic.fit(Xc)
    final_gmms_aic[cls] = gmm_aic

    # ----- Log-likelihood pura -----
    print(f"Addestro modello finale per classe {cls} con {k_ll} componenti (Log-Likelihood)...")
    gmm_ll = GaussianMixture(
        n_components=k_ll,
        covariance_type='full',
        random_state=42
    )
    gmm_ll.fit(Xc)
    final_gmms_ll[cls] = gmm_ll


Addestro modello finale per classe A con 1 componenti (BIC)...
Addestro modello finale per classe A con 2 componenti (AIC)...
Addestro modello finale per classe A con 3 componenti (Log-Likelihood)...
Addestro modello finale per classe B con 1 componenti (BIC)...
Addestro modello finale per classe B con 2 componenti (AIC)...
Addestro modello finale per classe B con 2 componenti (Log-Likelihood)...
Addestro modello finale per classe C con 1 componenti (BIC)...
Addestro modello finale per classe C con 2 componenti (AIC)...
Addestro modello finale per classe C con 3 componenti (Log-Likelihood)...
Addestro modello finale per classe D con 1 componenti (BIC)...
Addestro modello finale per classe D con 2 componenti (AIC)...
Addestro modello finale per classe D con 2 componenti (Log-Likelihood)...
Addestro modello finale per classe E con 1 componenti (BIC)...
Addestro modello finale per classe E con 2 componenti (AIC)...
Addestro modello finale per classe E con 4 componenti (Log-Likelihood)...


In [6]:
from sklearn.metrics import accuracy_score

# ===============================
# 6) PREDIZIONE SUL TEST
# ===============================

def predict_gmms(X, gmms, priors, classes):
    y_pred = []
    for x in X:
        class_scores = {}
        for cls in classes:
            log_like = gmms[cls].score(x.reshape(1, -1))
            log_post = log_like + np.log(priors[cls])
            class_scores[cls] = log_post
        y_pred.append(max(class_scores, key=class_scores.get))
    return np.array(y_pred)

# Predizione con BIC
y_pred_bic = predict_gmms(X_test, final_gmms_bic, priors, classes)
acc_bic = accuracy_score(y_test, y_pred_bic)
print(f"Accuracy sul test (BIC): {acc_bic*100:.2f}%")

# Predizione con AIC
y_pred_aic = predict_gmms(X_test, final_gmms_aic, priors, classes)
acc_aic = accuracy_score(y_test, y_pred_aic)
print(f"Accuracy sul test (AIC): {acc_aic*100:.2f}%")

# Predizione con Log-Likelihood pura
y_pred_ll = predict_gmms(X_test, final_gmms_ll, priors, classes)
acc_ll = accuracy_score(y_test, y_pred_ll)
print(f"Accuracy sul test (Log-Likelihood): {acc_ll*100:.2f}%")


Accuracy sul test (BIC): 77.57%
Accuracy sul test (AIC): 84.08%
Accuracy sul test (Log-Likelihood): 77.30%


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ===============================
# MONDRIAN CONFORMAL PREDICTION
# ===============================

# 1) Split test set into calibration and final test
print("Splitting test set into calibration and final test...")
X_calib, X_test_final, y_calib, y_test_final = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, stratify=y_test
)
print(f"Calibration set: {X_calib.shape[0]} samples")
print(f"Final test set: {X_test_final.shape[0]} samples\n")

# Define significance level (miscoverage rate)
alpha = 0.1  # Target: 90% coverage

# 2) Compute nonconformity scores on calibration set
# Nonconformity score = negative log-likelihood under the TRUE class
print("Computing nonconformity scores on calibration set...")
nonconformity_scores_calib = {cls: [] for cls in classes}

for i, x in enumerate(X_calib):
    true_class = y_calib[i]
    # Get the GMM for the true class (using AIC-selected models)
    gmm = final_gmms_aic[true_class]
    # Compute log-likelihood under true class
    log_like = gmm.score(x.reshape(1, -1))
    # Nonconformity score: negative log-likelihood (higher = less conforming)
    nonconf_score = -log_like
    nonconformity_scores_calib[true_class].append(nonconf_score)

# 3) Compute quantiles per class (Mondrian approach)
print("\nComputing per-class quantiles...")
quantiles = {}
for cls in classes:
    scores = np.array(nonconformity_scores_calib[cls])
    n_cls = len(scores)
    # Conformal quantile formula: ceil((n+1)(1-α))/n
    q_level = np.ceil((n_cls + 1) * (1 - alpha)) / n_cls
    q_level = min(q_level, 1.0)  # Cap at 1.0
    quantiles[cls] = np.quantile(scores, q_level)
    print(f"Class {cls}: n={n_cls}, quantile={quantiles[cls]:.4f}")

# 4) Build prediction sets on final test set
print("\n" + "="*70)
print("BUILDING PREDICTION SETS ON FINAL TEST SET")
print("="*70)

def predict_conformal_mondrian(X, gmms, classes, quantiles):
    """
    Mondrian Conformal Prediction for GMM
    Returns: point predictions and prediction sets
    """
    predictions = []
    prediction_sets = []
    
    for x in X:
        pred_set = []
        nonconf_scores = {}
        
        # Compute nonconformity score for each class
        for cls in classes:
            log_like = gmms[cls].score(x.reshape(1, -1))
            nonconf_score = -log_like
            nonconf_scores[cls] = nonconf_score
            
            # Include class in prediction set if score ≤ class-specific quantile
            if nonconf_score <= quantiles[cls]:
                pred_set.append(cls)
        
        # Handle edge case: empty prediction set
        if len(pred_set) == 0:
            pred_set = [min(nonconf_scores, key=nonconf_scores.get)]
        
        prediction_sets.append(pred_set)
        
        # Point prediction: class with minimum nonconformity (maximum likelihood)
        predictions.append(min(nonconf_scores, key=nonconf_scores.get))
    
    return np.array(predictions), prediction_sets

# Make predictions
y_pred_conf, pred_sets = predict_conformal_mondrian(
    X_test_final, final_gmms_aic, classes, quantiles
)

# ===============================
# 5) EVALUATION
# ===============================

print("\n" + "="*70)
print(f"MONDRIAN CONFORMAL PREDICTION RESULTS (α = {alpha})")
print("="*70)

# Accuracy (point prediction)
accuracy = accuracy_score(y_test_final, y_pred_conf)
print(f"\n1. ACCURACY (Point Prediction): {accuracy*100:.2f}%")

# Coverage (proportion of times true class is in prediction set)
coverage_list = [1 if y_test_final[i] in pred_sets[i] else 0 
                 for i in range(len(y_test_final))]
avg_coverage = np.mean(coverage_list)
print(f"\n2. COVERAGE:")
print(f"   Average Coverage: {avg_coverage*100:.2f}%")
print(f"   Target Coverage: {(1-alpha)*100:.0f}%")
print(f"   Coverage Gap: {(avg_coverage-(1-alpha))*100:.2f}%")

# Cardinality (size of prediction sets)
cardinalities = [len(pred_set) for pred_set in pred_sets]
avg_cardinality = np.mean(cardinalities)
print(f"\n3. CARDINALITY:")
print(f"   Average Set Size: {avg_cardinality:.2f}")
print(f"   Min Set Size: {min(cardinalities)}")
print(f"   Max Set Size: {max(cardinalities)}")
print(f"   Median Set Size: {np.median(cardinalities):.1f}")

# Per-class coverage
print(f"\n4. PER-CLASS COVERAGE:")
class_coverage = {}
for cls in classes:
    mask = y_test_final == cls
    if np.sum(mask) > 0:
        cls_coverage = np.mean([1 if cls in pred_sets[i] else 0 
                                for i in range(len(y_test_final)) if mask[i]])
        class_coverage[cls] = cls_coverage
        print(f"   Class {cls}: {cls_coverage*100:.1f}% (n={np.sum(mask)})")

# ===============================
# 6) DETAILED EXAMPLES
# ===============================

print("\n" + "="*70)
print("EXAMPLES OF PREDICTION SETS")
print("="*70)

n_examples = min(40, len(X_test_final))
for i in range(n_examples):
    true_class = y_test_final[i]
    pred_set = pred_sets[i]
    point_pred = y_pred_conf[i]
    in_set = "✓" if true_class in pred_sets[i] else "✗"
    correct = "✓" if point_pred == true_class else "✗"
    
    print(f"Sample {i+1:3d}: True={true_class} | Pred={point_pred} {correct} | "
          f"Set={sorted(pred_set)} | Size={len(pred_set)} | Covered={in_set}")

# Distribution of set sizes
print("\n" + "="*70)
print("DISTRIBUTION OF PREDICTION SET SIZES")
print("="*70)
from collections import Counter
size_dist = Counter(cardinalities)
for size in sorted(size_dist.keys()):
    count = size_dist[size]
    pct = count / len(cardinalities) * 100
    bar = "█" * int(pct / 2)
    print(f"Size {size:2d}: {count:4d} samples ({pct:5.1f}%) {bar}")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"✓ Point Prediction Accuracy: {accuracy*100:.2f}%")
print(f"✓ Average Coverage: {avg_coverage*100:.2f}% (Target: {(1-alpha)*100:.0f}%)")
print(f"✓ Average Set Size: {avg_cardinality:.2f}")
print(f"✓ Efficiency: {(avg_cardinality-1)/(len(classes)-1)*100:.1f}% of maximum")
print("="*70)

Splitting test set into calibration and final test...
Calibration set: 1476 samples
Final test set: 1476 samples

Computing nonconformity scores on calibration set...

Computing per-class quantiles...
Class A: n=58, quantile=50.8348
Class B: n=58, quantile=-55.1419
Class C: n=52, quantile=136.2534
Class D: n=58, quantile=16.4190
Class E: n=57, quantile=-4.4995
Class F: n=57, quantile=69.8946
Class G: n=58, quantile=810.7697
Class H: n=57, quantile=168.9263
Class I: n=57, quantile=-25.8738
Class J: n=58, quantile=232.2555
Class K: n=58, quantile=25.9349
Class L: n=57, quantile=195.6564
Class M: n=58, quantile=242.0287
Class N: n=57, quantile=36.9825
Class O: n=45, quantile=104.1786
Class P: n=58, quantile=33.4870
Class Q: n=58, quantile=114.2741
Class R: n=57, quantile=29.4266
Class S: n=58, quantile=-9.6692
Class T: n=57, quantile=63.1498
Class U: n=58, quantile=-16.7057
Class V: n=57, quantile=2.0168
Class W: n=57, quantile=-27.1243
Class X: n=57, quantile=195.4775
Class Y: n=57, quan